# ShieldVI V3 — AI + từ lóng Việt/Anh + lưu nhãn SQLite

Chạy từ trên xuống dưới. Notebook này là bản *mới*, đã thêm sẵn toàn bộ import (`Path`, `drive`, `files`, `time`, `subprocess`, `output`).

**Cảnh báo:** API không có đăng nhập, chỉ dùng trong Colab riêng / máy cục bộ. Không chia sẻ link cho người lạ. Không dùng thông tin cá nhân thật trong dữ liệu demo.

Cần file `ShieldVI_V3_Hybrid_Review.zip` và checkpoint ViSoBERT V1 trong Google Drive.


## 1 — Chỉ khi chưa có Transformers 4.57.3: cài một lần rồi Restart session

Colab có thể báo xung đột `gradio` hoặc `diffusers`: đây là các gói không dùng trong ShieldVI. Sau khi cài xong chọn **Runtime → Restart session**, rồi **bỏ qua ô cài đặt** khi chạy tiếp.


In [ ]:
%pip install -q "transformers==4.57.3" "huggingface-hub==0.36.2" "tokenizers>=0.22,<0.23" "fastapi>=0.115,<1" "uvicorn[standard]>=0.30,<1" "pydantic>=2.8,<3" "requests>=2.28" "pytest>=8"


## 2 — Import đầy đủ & kiểm tra phiên bản (chạy sau restart)


In [ ]:
import os, sys, json, time, shutil, zipfile, subprocess
from pathlib import Path
import requests
import transformers, torch
from google.colab import drive, files, output

print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
assert transformers.__version__ == "4.57.3", "Hãy cài 4.57.3, Restart session, rồi bỏ qua ô cài đặt"
print("Import và môi trường OK")


## 3 — Upload ZIP V3 và giải nén

Chỉ chọn file tên **`ShieldVI_V3_Hybrid_Review.zip`**. Không chọn ZIP V2 cũ.


In [ ]:
ROOT = Path("/content/shieldvi_v3")
ZIP_FILE = Path("/content/ShieldVI_V3_Hybrid_Review.zip")
if not ZIP_FILE.is_file():
    print("Chọn ShieldVI_V3_Hybrid_Review.zip")
    uploaded = files.upload()
    ZIP_FILE = Path("/content") / next(iter(uploaded))
assert ZIP_FILE.is_file(), ZIP_FILE
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_FILE) as z:
    z.extractall(ROOT)
assert (ROOT/'frontend/package.json').is_file(), "Sai file ZIP: thiếu frontend/package.json"
assert (ROOT/'app/lexical.py').is_file(), "Đây không phải ZIP V3: thiếu app/lexical.py"
assert (ROOT/'app/review_store.py').is_file(), "Sai ZIP: thiếu review_store.py"
print("Giải nén V3 thành công:",ROOT)


## 4 — Build React (cần mạng để tải package NPM)


In [ ]:
def run_command(cmd, cwd):
    print("Đang chạy:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print(completed.stdout[-2500:])
    if completed.returncode:
        print(completed.stderr[-3000:])
        raise RuntimeError("Lệnh thất bại: " + " ".join(cmd))

run_command(["npm","install","--no-audit","--no-fund"], ROOT/'frontend')
run_command(["npm","run","build"], ROOT/'frontend')
assert (ROOT/'app/static/index.html').is_file(), "React build chưa tạo index.html"
print("Đã build React V3 thành công!")


## 5 — Kết nối Google Drive cũ chứa ViSoBERT và đặt nơi lưu nhãn vĩnh viễn


In [ ]:
drive.mount('/content/drive')
MODEL_PATH = '/content/drive/MyDrive/visobert_toxic_final'
assert (Path(MODEL_PATH)/'config.json').is_file(), 'Sai tài khoản Drive hoặc MODEL_PATH; không thấy config.json'
assert (Path(MODEL_PATH)/'threshold_config.json').is_file(), 'Thiếu threshold_config.json'
REVIEW_DB_PATH = '/content/drive/MyDrive/shieldvi_v3_reviews.sqlite3'
print('Checkpoint OK:', MODEL_PATH)
print('Nhãn thủ công lưu tại:', REVIEW_DB_PATH)


## 6 — Chạy FastAPI (cùng origin với React)

Nếu phiên Colab cũ còn web server ở cổng 8000, hãy ngừng server cũ hoặc Restart session trước khi chạy lại bước này. Notebook tự nhận diện server V3 sẵn có.


In [ ]:
URL='http://127.0.0.1:8000'
def check_server():
    try:
        resp=requests.get(URL+'/openapi.json',timeout=2)
        if resp.status_code==200:
            return resp.json().get('info',{}).get('version')
    except requests.RequestException:
        return None
    return None

running_version=check_server()
if running_version and running_version!='0.3.0':
    raise RuntimeError(f'Cổng 8000 đang chạy backend cũ {running_version}. Hãy Restart session rồi chạy lại từ phần 2.')

if running_version!='0.3.0':
    LOG=ROOT/'uvicorn_colab.log'
    env=os.environ.copy()
    env['MODEL_PATH']=MODEL_PATH
    env['REVIEW_DB_PATH']=REVIEW_DB_PATH
    logfile=open(LOG,'w',encoding='utf-8')
    server=subprocess.Popen([sys.executable,'-m','uvicorn','app.main:app','--host','127.0.0.1','--port','8000'],
                            cwd=ROOT,env=env,stdout=logfile,stderr=subprocess.STDOUT)
    for attempt in range(90):
        try:
            r=requests.get(URL+'/health',timeout=4)
            if r.status_code==200 and r.json().get('model_loaded'):
                break
        except requests.RequestException:
            pass
        if server.poll() is not None:
            raise RuntimeError('Backend dừng; log: '+LOG.read_text(encoding='utf-8')[-3000:])
        time.sleep(2)
    else:
        raise RuntimeError('ViSoBERT chưa sẵn sàng; log: '+LOG.read_text(encoding='utf-8')[-3000:])
else:
    r=requests.get(URL+'/health',timeout=5)

assert r.json().get('model_loaded'), r.json()
print('Health:',r.status_code,r.json())
assert requests.get(URL+'/dashboard/',timeout=10).status_code==200
print('Dashboard: HTTP 200')


## 7 — Test từ lóng & API SQLite với kịch bản trộn tiếng Việt và tiếng Anh


In [ ]:
payload=json.loads((ROOT/'examples/DEMO_HYBRID_TEENCODE_EN_VI.json').read_text(encoding='utf-8'))
r=requests.post(URL+'/api/analyze',json=payload,timeout=120)
assert r.status_code==200,r.text
result=r.json()
print('Analyze HTTP:',r.status_code,'| Bình luận:',len(result['comments']))
for item in result['comments']:
    if item['lexical_signals']:
        print(item['comment_id'],'| model_score:',round(item['toxicity_score'],3),
              '| lexical:',[x['matched_form'] for x in item['lexical_signals']],
              '| directed_candidate:',item['directed_insult_candidate'])
print('Review stats:',requests.get(URL+'/api/reviews/stats',timeout=10).json())


## 8 — Mở giao diện và thử **Lưu vào SQLite**

Trong website → Nhập JSON → chọn `DEMO_HYBRID_TEENCODE_EN_VI.json` → Phân tích → Xác nhận thủ công cho vài bình luận → **Lưu vào SQLite** → thử **Tải nhãn đã lưu** → vào **Thống kê kiểm duyệt**.


In [ ]:
try:
    output.serve_kernel_port_as_iframe(8000,path='/dashboard/',height=900)
except TypeError:
    link=output.eval_js('google.colab.kernel.proxyPort(8000)')
    print('Mở dashboard:',link.rstrip('/')+'/dashboard/')


## 9 — Sao lưu nhãn đã lưu (tùy chọn)

Nhãn nằm trong `MyDrive/shieldvi_v3_reviews.sqlite3`. Website còn có nút Xuất SQLite CSV ở trang Thống kê kiểm duyệt. Không chia sẻ file database chứa ID tài khoản thật.


In [ ]:
from google.colab import files as colab_files
if Path(REVIEW_DB_PATH).is_file():
    print('Database hiện tại:', REVIEW_DB_PATH)
    # Uncomment the next line if you want a downloadable backup:
    # colab_files.download(REVIEW_DB_PATH)
else:
    print('Chưa có bản ghi kiểm duyệt, hãy lưu nhãn trong dashboard trước.')
